# Modèle de régression linéaire

Ce notebook charge `df_final` préparé dans `model_testing.ipynb`, puis entraîne et évalue la régression. Les coefficients sont disponibles dans `df_coefficients`, les prédictions de test dans `df_predictions` et les métriques dans `validation_results`.

Aucun fichier CSV supplémentaire n'est créé. Exécuter les cellules dans l'ordre.

In [ ]:
%run ./model_testing.ipynb

## Régression linéaire — prédire la hausse du loyer effectif

**Y = `effective_growth_pct`**, en points de pourcentage de croissance annualisée. Le modèle est une régression linéaire ordinaire, avec variables catégorielles encodées et variables numériques standardisées.

Les prédicteurs sont le loyer effectif précédent, l'intervalle prévu entre les baux, le marché, le type de nouveau bail et les trois facteurs publics de **l'année précédente**. Le type de bail et l'intervalle doivent être connus ou fixés comme hypothèses à la date de prévision.

`effective_rent` est exclu : avec `prev_effective_rent` et `gap_years`, il détermine exactement Y. Les concessions du nouveau bail sont aussi exclues de cette prévision; elles ne sont pas connues à l'avance. Les identifiants d'appartements ne sont pas utilisés comme valeurs numériques.

Le décalage des facteurs publics évite d'utiliser les moyennes futures de l'année prédite. Il faut toutefois vérifier les dates de publication et les révisions pour une simulation strictement fidèle aux informations disponibles à l'époque.

Un modèle commun est conservé. `Market` et `type_bail` restent des features catégorielles; `bedrooms` est ajouté aux features numériques. Les seuls facteurs publics retenus sont `general_cpi`, `vacancy_rate` et `unemployement_rate`, décalés d’un an.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

TARGET = "effective_growth_pct"
public_features = ["general_cpi", "vacancy_rate", "unemployement_rate"]
# Repartir des attributs privés de df_final, puis rattacher les facteurs à Year - 1.
regression_data = df_final.drop(columns=public_features).copy()
public_lagged = df_public[["Market", "Year"] + public_features].copy()
public_lagged["public_source_year"] = public_lagged.Year
public_lagged["Year"] = public_lagged.Year + 1
public_lagged = public_lagged.rename(columns={c: "lag1_" + c for c in public_features})
regression_data = regression_data.merge(public_lagged, on=["Market", "Year"], how="left", validate="many_to_one")
NUMERIC_FEATURES = ["prev_effective_rent", "gap_years", "bedrooms"] + ["lag1_" + c for c in public_features]
CATEGORICAL_FEATURES = ["Market", "type_bail"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
eligible = (regression_data[FEATURES + [TARGET]].notna().all(axis=1)
            & np.isfinite(regression_data[NUMERIC_FEATURES + [TARGET]].astype(float)).all(axis=1))
model_data = regression_data.loc[eligible].copy()
print("Paires disponibles :", len(regression_data), "; paires utilisables :", len(model_data))
print("Les paires sans facteurs de l'année précédente sont exclues, pas imputées à zéro.")
display(model_data.groupby(["Market", "Year"]).size().rename("nombre_paires").to_frame())
TEST_YEAR = 2025
train = model_data.loc[model_data.Year.lt(TEST_YEAR)].copy()
test = model_data.loc[model_data.Year.eq(TEST_YEAR)].copy()
assert not train.empty and not test.empty
assert train.Year.max() < test.Year.min()
assert set(test.Market).issubset(set(train.Market))

# Ne pas inclure Year : le modèle utilise les facteurs économiques et le marché.
def make_linear_model():
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("category", OneHotEncoder(drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([("preprocessing", preprocessing), ("regression", LinearRegression())])

linear_model = make_linear_model()
linear_model.fit(train[FEATURES], train[TARGET])
train_prediction = linear_model.predict(train[FEATURES])
test_prediction = linear_model.predict(test[FEATURES])
# Référence simple calculée sur l'entraînement uniquement.
baseline = np.full(len(test), train[TARGET].mean())
validation_results = pd.DataFrame([
    {"modele": "Régression linéaire", "MAE_points": mean_absolute_error(test[TARGET], test_prediction),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], test_prediction)), "R2": r2_score(test[TARGET], test_prediction)},
    {"modele": "Moyenne historique", "MAE_points": mean_absolute_error(test[TARGET], baseline),
     "RMSE_points": np.sqrt(mean_squared_error(test[TARGET], baseline)), "R2": r2_score(test[TARGET], baseline)},
]).set_index("modele")
display(validation_results.round(4))
df_predictions = test[["sPropCode", "sUnitCode", "Market", "Year", "type_bail", TARGET]].copy()
df_predictions["predicted_growth_pct"] = test_prediction
df_predictions["error_points"] = df_predictions.predicted_growth_pct - df_predictions[TARGET]
display(df_predictions.head(15).round(3))

### Coefficients de chaque feature

Deux lectures sont fournies : le coefficient standardisé correspond à une augmentation d'un écart-type du prédicteur numérique; le coefficient dans l'unité originale correspond à +1 unité. Dans les deux cas, l'effet est exprimé en **points de pourcentage de hausse prédite**, toutes les autres variables du modèle étant maintenues constantes. Les variables catégorielles sont comparées à la catégorie de référence.

Le modèle est ajusté sur les années antérieures à 2025 : les coefficients affichés ne sont pas réajustés sur l'année de test.

In [ ]:
preprocessor = linear_model.named_steps["preprocessing"]
regressor = linear_model.named_steps["regression"]
scaler = preprocessor.named_transformers_["numeric"]
encoder = preprocessor.named_transformers_["category"]
numeric_coefficients = regressor.coef_[:len(NUMERIC_FEATURES)]
coefficient_rows = []
for feature, standardized, scale in zip(NUMERIC_FEATURES, numeric_coefficients, scaler.scale_):
    coefficient_rows.append({"feature": feature,
                             "coefficient_standardise": standardized,
                             "coefficient_unite_originale": standardized / scale,
                             "interpretation": "+1 unité du facteur; facteurs publics de l'année précédente"})
category_names = encoder.get_feature_names_out(CATEGORICAL_FEATURES)
for feature, coefficient in zip(category_names, regressor.coef_[len(NUMERIC_FEATURES):]):
    coefficient_rows.append({"feature": feature, "coefficient_standardise": coefficient,
                             "coefficient_unite_originale": coefficient,
                             "interpretation": "Écart par rapport à la catégorie de référence"})
df_coefficients = pd.DataFrame(coefficient_rows)
intercept_original = regressor.intercept_ - np.sum(numeric_coefficients * scaler.mean_ / scaler.scale_)
display(df_coefficients.round(6))
print("Constante dans les unités originales :", round(float(intercept_original), 6))
for name, categories in zip(CATEGORICAL_FEATURES, encoder.categories_):
    print(f"Catégorie de référence pour {name} : {categories[0]}")

# Vérifier que les coefficients dans les unités originales reproduisent les prédictions.
reconstructed = (intercept_original + test[NUMERIC_FEATURES].to_numpy(dtype=float)
                 @ (numeric_coefficients / scaler.scale_)
                 + encoder.transform(test[CATEGORICAL_FEATURES]) @ regressor.coef_[len(NUMERIC_FEATURES):])
assert np.allclose(reconstructed, test_prediction)

matrix = preprocessor.transform(train[FEATURES])
design = np.column_stack([np.ones(len(matrix)), matrix])
rank = np.linalg.matrix_rank(design)
number_parameters = design.shape[1]
model_identifiable = rank == number_parameters
print(f"Rang de la matrice : {rank}/{number_parameters}; groupes marché/année d'entraînement : {len(train[['Market','Year']].drop_duplicates())}")
if not model_identifiable:
    print("ATTENTION : les coefficients ne sont pas identifiables séparément. Plusieurs combinaisons donnent les mêmes prédictions d'entraînement.")
print("Les facteurs publics sont répétés pour les baux d'un même marché/année; leur nombre d'observations indépendantes est limité.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(test[TARGET], test_prediction, alpha=0.25, s=15)
low = min(test[TARGET].min(), test_prediction.min())
high = max(test[TARGET].max(), test_prediction.max())
axes[0].plot([low, high], [low, high], "--", color="gray")
axes[0].set(title="Test 2025 : hausse observée et prédite", xlabel="Hausse effective observée (%)", ylabel="Hausse effective prédite (%)")
coeff_plot = df_coefficients.set_index("feature").coefficient_standardise
coeff_plot.plot.barh(ax=axes[1])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set(title="Coefficients standardisés — modèle avant 2025", xlabel="Points de hausse par écart-type / catégorie", ylabel="")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Prévoir un nouveau bail avec des hypothèses explicites

La fonction ci-dessous utilise le modèle de validation et les facteurs publics de l'année précédente. Elle ne l'entraîne pas à nouveau. Fournir le loyer effectif précédent, le marché, le type de bail et l'intervalle prévu. Pour une utilisation finale en 2026, on pourra réajuster le modèle sur les données jusqu'en 2025 après avoir évalué la validation.

Les coefficients ne démontrent pas des causes. Si la matrice est de rang insuffisant, ne pas interpréter séparément les effets des trois facteurs publics : il faut davantage d'années ou réduire leur nombre. Un R² négatif sur le test signifie que les prédictions sont moins bonnes que la moyenne de Y dans ce test; la MAE et la référence historique permettent d'évaluer l'utilité pratique du modèle.

In [ ]:
def predict_rent_growth(prev_effective_rent, market, type_bail, year, bedrooms, gap_years=1.0):
    if prev_effective_rent <= 0 or gap_years <= 0:
        raise ValueError("Le loyer précédent et la durée doivent être positifs.")
    source = df_public.loc[df_public.Market.eq(market) & df_public.Year.eq(year - 1)]
    if len(source) != 1:
        raise ValueError("Facteurs publics de l'année précédente absents ou ambigus.")
    row = {"prev_effective_rent": prev_effective_rent, "gap_years": gap_years,
           "Market": market, "type_bail": type_bail, "bedrooms": bedrooms}
    row.update({"lag1_" + col: source.iloc[0][col] for col in public_features})
    scenario = pd.DataFrame([row])[FEATURES]
    if scenario.isna().any().any():
        raise ValueError("Un facteur nécessaire est manquant.")
    return float(linear_model.predict(scenario)[0])

# Exemple à adapter, sans confondre ce scénario avec une prévision du portefeuille :
# predict_rent_growth(2000, "Montréal", "Renouvellement", year=2026, bedrooms=2)


## Prévision finale 2026

Après la validation sur 2025, nous ajustons un **nouveau modèle** sur toutes les paires admissibles jusqu'en 2025. Le modèle de validation et ses métriques restent distincts.

Hypothèses pour 2026 :

- Les facteurs publics utilisés sont ceux de 2025, puisque le modèle les décale d'un an.
- Le loyer effectif de référence est celui du dernier bail connu de chaque appartement au 31 décembre 2025.
- L'intervalle prévu entre deux baux est fixé à **1 an**. Il s'agit d'un scénario annualisé pour les appartements de l'extrait, pas d'un calendrier réel de renouvellements.
- Nous calculons deux scénarios par appartement : renouvellement et relocation. La part prévue de relocations est estimée avec les paires comparables de 2025 de son marché. Cette part est une hypothèse modifiable.
- Chaque appartement de l'extrait a le même poids. Le résultat est une hausse moyenne prédite dans l'échantillon; il ne mesure ni les recettes ni l'occupation du portefeuille réel.

Le modèle ne prévoit pas explicitement les promotions de 2026 : il prévoit la variation du loyer effectif à partir de l'historique. Ses limites de validation et de stabilité des coefficients restent applicables.

In [ ]:
FORECAST_YEAR = 2026
final_training_data = model_data.loc[model_data.Year.le(2025)].copy()
assert final_training_data.Year.max() == 2025
final_model = make_linear_model()
final_model.fit(final_training_data[FEATURES], final_training_data[TARGET])
print(f"Modèle final ajusté sur {len(final_training_data):,} paires de {int(final_training_data.Year.min())} à 2025.")

# Coefficients du modèle final, distincts de ceux du modèle de validation.
final_preprocessor = final_model.named_steps["preprocessing"]
final_regressor = final_model.named_steps["regression"]
final_scaler = final_preprocessor.named_transformers_["numeric"]
final_encoder = final_preprocessor.named_transformers_["category"]
final_numeric_coef = final_regressor.coef_[:len(NUMERIC_FEATURES)]
final_coefficient_rows = [
    {"feature": feature, "coefficient_standardise": coef, "coefficient_unite_originale": coef / scale}
    for feature, coef, scale in zip(NUMERIC_FEATURES, final_numeric_coef, final_scaler.scale_)
]
final_coefficient_rows += [
    {"feature": feature, "coefficient_standardise": coef, "coefficient_unite_originale": coef}
    for feature, coef in zip(final_encoder.get_feature_names_out(CATEGORICAL_FEATURES),
                             final_regressor.coef_[len(NUMERIC_FEATURES):])
]
df_coefficients_2026 = pd.DataFrame(final_coefficient_rows)
display(df_coefficients_2026.round(6))
final_matrix = final_preprocessor.transform(final_training_data[FEATURES])
final_design = np.column_stack([np.ones(len(final_matrix)), final_matrix])
print("Rang de la matrice finale :", np.linalg.matrix_rank(final_design), "/", final_design.shape[1])

In [ ]:
# Tous les appartements ayant un bail connu, pas uniquement ceux avec une paire récente.
CUTOFF = pd.Timestamp("2025-12-31")
known_leases = private_leases.loc[private_leases.lease_start.le(CUTOFF)].copy()
latest = known_leases.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY, as_index=False).tail(1).copy()
latest["Market"] = latest.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
latest = latest.loc[latest.effective_rent.gt(0) & np.isfinite(latest.effective_rent) & latest.Market.notna()].copy()
assert not latest.duplicated(UNIT_KEY).any()

# Proportions issues des paires 2025, calculées séparément par marché.
recent = df_final.loc[df_final.Year.eq(2025) & df_final.type_bail.isin(["Relocation", "Renouvellement"])]
relocation_share_2026 = recent.assign(relocation=recent.type_bail.eq("Relocation")).groupby("Market").relocation.mean().to_dict()
print("Hypothèse de part des relocations par marché :", relocation_share_2026)
# Pour modifier l'hypothèse : relocation_share_2026["Montréal"] = 0.30

scenario_parts = []
for kind in ["Renouvellement", "Relocation"]:
    scenario = latest[UNIT_KEY + ["sBuilding", "Market", "effective_rent", "lease_start", "sBeds"]].copy()
    scenario = scenario.rename(columns={"effective_rent": "prev_effective_rent", "lease_start": "reference_lease_start", "sBeds": "bedrooms"})
    scenario["Year"] = FORECAST_YEAR
    scenario["gap_years"] = 1.0
    scenario["type_bail"] = kind
    scenario_parts.append(scenario)
scenarios_2026 = pd.concat(scenario_parts, ignore_index=True)
source_2025 = df_public.loc[df_public.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
scenarios_2026 = scenarios_2026.merge(source_2025, on="Market", how="left", validate="many_to_one")
assert scenarios_2026[FEATURES].notna().all().all()
assert len(scenarios_2026) == 2 * len(latest)
scenarios_2026["predicted_growth_pct"] = final_model.predict(scenarios_2026[FEATURES])
scenarios_2026["predicted_effective_rent"] = scenarios_2026.prev_effective_rent * (
    1 + scenarios_2026.predicted_growth_pct / 100
) ** scenarios_2026.gap_years
assert scenarios_2026.predicted_effective_rent.gt(0).all()
display(scenarios_2026.groupby(["Market", "type_bail"]).agg(
    nombre_appartements=("predicted_growth_pct", "size"),
    hausse_moyenne_pct=("predicted_growth_pct", "mean"),
    hausse_mediane_pct=("predicted_growth_pct", "median"),
).round(3))

In [ ]:
def combine_2026_scenarios(scenarios, relocation_shares):
    result = scenarios.copy()
    shares = result.Market.map(relocation_shares)
    if shares.isna().any() or not shares.between(0, 1).all():
        raise ValueError("Chaque marché doit avoir une proportion de relocations entre 0 et 1.")
    result["scenario_weight"] = np.where(result.type_bail.eq("Relocation"), shares, 1 - shares)
    result["weighted_growth"] = result.predicted_growth_pct * result.scenario_weight
    unit_forecasts = result.groupby(UNIT_KEY + ["Market", "sBuilding"], as_index=False).agg(
        predicted_growth_pct=("weighted_growth", "sum"), total_weight=("scenario_weight", "sum")
    )
    assert np.allclose(unit_forecasts.total_weight, 1)
    return unit_forecasts.drop(columns="total_weight")

def estimate_2026(relocation_shares=None):
    """Hausse effective moyenne prévue en 2026 (%) pour les appartements de l'extrait."""
    shares = relocation_share_2026 if relocation_shares is None else relocation_shares
    return float(combine_2026_scenarios(scenarios_2026, shares).predicted_growth_pct.mean())

df_forecast_2026 = combine_2026_scenarios(scenarios_2026, relocation_share_2026)
forecast_2026_pct = estimate_2026()
print(f"HAUSSE EFFECTIVE MOYENNE PRÉVUE EN 2026 : {forecast_2026_pct:.2f} %")
print(f"Scénario fondé sur {len(df_forecast_2026):,} appartements de l'extrait, pondérés également.")
display(df_forecast_2026.groupby("Market").agg(
    nombre_appartements=("predicted_growth_pct", "size"), hausse_moyenne_pct=("predicted_growth_pct", "mean")
).round(3))
display(df_forecast_2026.head(15).round(3))

# Sensibilité de la moyenne prévue au choix des poids renouvellement/relocation.
sensitivity_2026 = pd.DataFrame([
    {"scenario": "Tous renouvellements", "hausse_pct": estimate_2026({m: 0 for m in relocation_share_2026})},
    {"scenario": "Parts observées en 2025", "hausse_pct": forecast_2026_pct},
    {"scenario": "Toutes relocations", "hausse_pct": estimate_2026({m: 1 for m in relocation_share_2026})},
])
display(sensitivity_2026.round(3))

In [ ]:
def backtest(target_year=2025):
    """Entraîne seulement avant target_year et évalue les baux de target_year."""
    history = model_data.loc[model_data.Year.lt(target_year)]
    held_out = model_data.loc[model_data.Year.eq(target_year)]
    if history.empty or held_out.empty:
        raise ValueError("Historique ou année test absents.")
    if not set(held_out.Market).issubset(set(history.Market)):
        raise ValueError("Un marché du test n'est pas présent dans l'historique.")
    fitted = make_linear_model().fit(history[FEATURES], history[TARGET])
    predicted = fitted.predict(held_out[FEATURES])
    return pd.Series({"test_year": target_year, "nombre_paires": len(held_out),
                      "R2": r2_score(held_out[TARGET], predicted),
                      "MAE_points": mean_absolute_error(held_out[TARGET], predicted),
                      "RMSE_points": np.sqrt(mean_squared_error(held_out[TARGET], predicted))})

# Les métriques précédentes ont déjà évalué 2025 : vérifier la cohérence, sans réentraîner.
assert np.isclose(forecast_2026_pct, df_forecast_2026.predicted_growth_pct.mean())


### Lire le résultat final

`forecast_2026_pct` est la prévision moyenne de hausse effective, sous les hypothèses ci-dessus. `df_forecast_2026` contient les prévisions par appartement; `df_coefficients_2026` les coefficients du modèle réajusté.

Les métriques R², MAE et RMSE restent celles du **test 2025**, pas celles de 2026 : les résultats réels de 2026 ne sont pas dans ces fichiers. La faible performance du test et le manque d'années indépendantes rendent la prévision exploratoire. Le scénario annuel ne prétend pas décrire les baux effectivement signés en 2026.

## Fonctions correspondant au gabarit du travail

Les fonctions finales ci-dessous remplacent les signatures simplifiées précédentes. Elles acceptent les baux bruts ou renommés et refont la préparation à chaque appel. `estimate_2026` renvoie un pourcentage; `backtest` renvoie les hausses moyenne prédite/observée, leur écart et les métriques individuelles sur l'année cible.

Définition : hausse annualisée moyenne du loyer effectif à appartement constant, renouvellements et relocations combinés. Pour la prévision, leurs proportions sont celles de la dernière année de paires disponibles par marché. Les facteurs publics sont décalés d'un an. `asking` est accepté pour respecter le gabarit mais n'entre pas dans cette méthode; les loyers affichés ne sont pas la cible retenue.

`external` peut recevoir une table publique avec `Market`, `Year` et les six facteurs. En son absence, la table publique chargée dans ce notebook est utilisée. `backtest` utilise cette même table, limitée aux années antérieures à la cible. Les fichiers étant des séries révisées et sans dates de publication, la simulation reste une approximation des informations disponibles à l'époque.

In [ ]:
def prepare_template_leases(leases):
    data = leases.copy()
    if "bedrooms" not in data:
        data["bedrooms"] = pd.to_numeric(data.sBeds, errors="raise")
    data = data.rename(columns={k: v for k, v in LEASE_COLUMN_RENAMES.items() if k in data and v not in data})
    data["lease_start"] = pd.to_datetime(data.lease_start)
    data["lease_start_year"] = data.lease_start.dt.year
    data["Market"] = data.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
    data["Year"] = data.lease_start_year
    data["type_bail"] = data.is_renewal.map({0: "Relocation", 1: "Renouvellement"})
    return data


def prepare_template_pairs(leases, external):
    pairs = same_unit_growth(leases, "effective_rent")
    pairs[TARGET] = pairs.growth_pct
    public = external[["Market", "Year"] + public_features].copy()
    assert not public.duplicated(["Market", "Year"]).any()
    public["Year"] += 1
    public = public.rename(columns={c: "lag1_" + c for c in public_features})
    pairs = pairs.merge(public, on=["Market", "Year"], how="left", validate="many_to_one")
    return pairs.dropna(subset=FEATURES + [TARGET])


def fit_template_model(leases, year, external):
    history = prepare_template_leases(leases)
    history = history.loc[history.lease_start.lt(pd.Timestamp(year=year, month=1, day=1))].copy()
    public_history = external.loc[external.Year.lt(year)].copy()
    training = prepare_template_pairs(history, public_history)
    if training.empty:
        raise ValueError("Pas de paires et de facteurs publics disponibles avant l'année cible.")
    # Catégories définies à l'avance; un marché sans historique est identifié dans le backtest.
    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("category", OneHotEncoder(categories=[MARKETS, ["Relocation", "Renouvellement"]],
                                   drop="first", handle_unknown="error", sparse_output=False), CATEGORICAL_FEATURES),
    ])
    fitted = Pipeline([("preprocessing", preprocessing), ("regression", LinearRegression())])
    fitted.fit(training[FEATURES], training[TARGET])
    return fitted, history, training, public_history


def estimate_2026(leases, asking, external=None):
    """Hausse effective annualisée moyenne prévue en 2026 (%), mêmes appartements."""
    public = df_public if external is None else external
    fitted, history, training, public_history = fit_template_model(leases, 2026, public)
    latest = history.sort_values(UNIT_KEY + ["lease_start"]).groupby(UNIT_KEY).tail(1).copy()
    latest = latest.loc[latest.effective_rent.gt(0) & latest.Market.notna()]
    parts = []
    # Part de relocations de la dernière année de paires utilisables dans chaque marché.
    shares = {}
    for market, group in training.groupby("Market"):
        recent = group.loc[group.Year.eq(group.Year.max())]
        shares[market] = recent.type_bail.eq("Relocation").mean()
    for kind in ["Renouvellement", "Relocation"]:
        scenario = latest[UNIT_KEY + ["Market", "effective_rent", "bedrooms"]].rename(columns={"effective_rent": "prev_effective_rent"}).copy()
        scenario["gap_years"] = 1.0
        scenario["type_bail"] = kind
        source = public_history.loc[public_history.Year.eq(2025), ["Market"] + public_features].rename(columns={c: "lag1_" + c for c in public_features})
        scenario = scenario.merge(source, on="Market", how="left", validate="many_to_one")
        if scenario[FEATURES].isna().any().any():
            raise ValueError("Facteurs publics 2025 ou caractéristiques de scénario manquants.")
        share = scenario.Market.map(shares)
        if share.isna().any():
            raise ValueError("Aucun historique pour estimer le poids d'un marché.")
        scenario["weight"] = share if kind == "Relocation" else 1 - share
        scenario["weighted_growth"] = fitted.predict(scenario[FEATURES]) * scenario.weight
        parts.append(scenario)
    unit_growth = pd.concat(parts).groupby(UNIT_KEY).weighted_growth.sum()
    return float(unit_growth.mean())


def backtest(leases, target_year):
    """Entraîne avant target_year et compare aux paires réelles de cette année."""
    fitted, history, training, public_history = fit_template_model(leases, target_year, df_public)
    # Les baux de l'année cible ne servent qu'à mesurer Y après l'entraînement.
    evaluation_leases = prepare_template_leases(leases)
    evaluation_leases = evaluation_leases.loc[evaluation_leases.lease_start.lt(pd.Timestamp(year=target_year + 1, month=1, day=1))]
    evaluation = prepare_template_pairs(evaluation_leases, public_history)
    evaluation = evaluation.loc[evaluation.Year.eq(target_year)]
    if evaluation.empty:
        raise ValueError("Aucune paire évaluable pour l'année cible.")
    predicted = fitted.predict(evaluation[FEATURES])
    unseen_markets = sorted(set(evaluation.Market) - set(training.Market))
    return {
        "annee": target_year, "nombre_paires": len(evaluation),
        "hausse_predite_moyenne_pct": float(predicted.mean()),
        "hausse_observee_moyenne_pct": float(evaluation[TARGET].mean()),
        "erreur_moyenne_points": float(predicted.mean() - evaluation[TARGET].mean()),
        "R2": r2_score(evaluation[TARGET], predicted),
        "MAE_points": mean_absolute_error(evaluation[TARGET], predicted),
        "RMSE_points": np.sqrt(mean_squared_error(evaluation[TARGET], predicted)),
        "marches_sans_historique": ", ".join(unseen_markets),
    }

# Les baux renommés préparés dans model_testing peuvent être passés directement.
asking = pd.read_csv("private_data/equinoxe_asking_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
estimation_gabarit_2026 = estimate_2026(private_leases, asking, external=df_public)
print(f"Estimation 2026 : {estimation_gabarit_2026:.2f} %")
backtest_results = pd.DataFrame([backtest(private_leases, year) for year in (2023, 2024, 2025)])
display(backtest_results.round(4))
assert np.isclose(estimation_gabarit_2026, forecast_2026_pct)
